# KL-anchor validation on Kev-9B — one run: `kl_anchor_alpha = 0.005` (`wu_table_alpha=0.005`)

Same settings as `00-validate-kl-anchor-9b.ipynb` (Wu table only, λ = 1, `PEAK_LR = 5e-6`, 3 epochs); one α per notebook so each runs as its own job, saved under `runs/kl_anchor_9b/`.

In [ ]:
import os
import sys
import time
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (c for p in (Path.cwd(), *Path.cwd().parents) for c in (p, p / "code") if all((c / d).is_dir() for d in ("kev", "data", "models"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and models/ at or above {Path.cwd()}")

PAPER_DIR = CODE_DIR / "papers" / "beta-loss"
SRC_DIR = PAPER_DIR / "src"
if not (SRC_DIR / "paper_beta").is_dir():
    raise RuntimeError(f"`paper_beta` package not found in {SRC_DIR}")

# Run from `code/` so data loaders and shared disk caches resolve identically to all beta-training notebooks
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev"), str(SRC_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from models.kev_pointer import KevPointerModel
from training import free_gpu

import beta_ols
from beta_ols import core as beta_core
from beta_train.diagnostics import find_max_micro_chunk
from beta_train.items import build_items_cached, items_summary
from beta_train.model_state import set_checkpointing, snapshot_trainable
from beta_train.validation import PopulationValidator, validate_all_cached, validation_metrics_frame
from paper_beta import (
    PaperTrainConfig,
    run_paper_training,
)

# Shared disk cache for tokenized items and untrained step-0 validation; isolated run directory for alpha validation
CACHE_DIR = PAPER_DIR / ".cache" / "beta_training"
RUNS_DIR = PAPER_DIR / "runs" / "kl_anchor_9b"

def alpha_slug(alpha: float) -> str:
    return f"wu_table_alpha={alpha:g}"


print(f"CODE_DIR : {CODE_DIR}")
print(f"PAPER_DIR: {PAPER_DIR}")
print(f"RUNS_DIR : {RUNS_DIR}")

In [ ]:
# Require CUDA and release any previously loaded `kev_model` before allocating a fresh 9B backbone
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required to train jaredpalmer/kev-9b in bf16")
if "kev_model" in globals():
    globals().pop("kev_model")
device = "cuda"
free_gpu()
torch.manual_seed(42)

KEV_BASE = "jaredpalmer/kev-9b@db029f08b290afd9fee4aa4bbcd9ae48602d1eb0"
LORA_RANK = 64
WEIGHTS_DTYPE = "bf16"
HEAD_TEMPERATURE = 1.0
GRADIENT_CHECKPOINTING = False

t_model0 = time.perf_counter()
kev_model = KevPointerModel.build(
    KEV_BASE,
    device=device,
    lora_rank=LORA_RANK,
    weights_dtype=WEIGHTS_DTYPE,
)
kev_model.net.head.temperature = HEAD_TEMPERATURE
kev_model.train()
set_checkpointing(kev_model, GRADIENT_CHECKPOINTING)
torch.cuda.synchronize()
model_load_sec = time.perf_counter() - t_model0

trainable_params_count = sum(p.numel() for p in kev_model.trainable_parameters())
total_params_count = sum(p.numel() for p in kev_model.parameters())
display(
    pd.Series(
        {
            "base_model": KEV_BASE,
            "mode": kev_model.mode,
            "training": kev_model.training,
            "weights_dtype": kev_model.meta.weights_dtype,
            "lora_rank": kev_model.meta.lora,
            "trainable_params": f"{trainable_params_count:,}",
            "total_params": f"{total_params_count:,}",
            "gradient_checkpointing": GRADIENT_CHECKPOINTING,
            "model_load_sec": round(model_load_sec, 3),
        },
        name="kev_9b_train_mode",
    ).to_frame()
)

INIT_TRAINABLE_STATE = snapshot_trainable(kev_model)
MODEL_KEY = {
    "base": KEV_BASE,
    "lora_rank": LORA_RANK,
    "weights_dtype": WEIGHTS_DTYPE,
    "head_temperature": HEAD_TEMPERATURE,
}
print(f"Snapshotted {len(INIT_TRAINABLE_STATE)} trainable tensors")

# 1. `wu2025` Fitted OLS Targets & Held-Out Datasets (`kriner2026`, `grummon2025`)

In [ ]:
ITEM_DATASETS = ["wu2025"]
VALIDATION_DATASETS = ["wu2025", "kriner2026", 'grummon2025']
LOADED_DATASETS = [*ITEM_DATASETS, *VALIDATION_DATASETS]

choice_dfs: dict[str, pd.DataFrame] = {}
regression_rows_by_dataset: dict[str, pd.DataFrame] = {}
regression_specs_by_dataset: dict[str, dict] = {}
human_ols_by_dataset: dict[str, pd.DataFrame] = {}
module_by_dataset = {}
piece_names_by_study: dict[str, tuple[str, ...]] = {}

for dataset_name in LOADED_DATASETS:
    dataset_module = beta_ols.DATASET_MODULES[dataset_name]
    full_choice_df = dataset_module.load_choice_df()
    pieces = {}
    if dataset_name in VALIDATION_DATASETS:
        pieces[dataset_name] = full_choice_df
    if dataset_name in ITEM_DATASETS:
        task_pieces = beta_core.split_by_task(full_choice_df, dataset_name)
        pieces.update(task_pieces)
        piece_names_by_study[dataset_name] = tuple(task_pieces)

    for piece_name, choice_df in pieces.items():
        rows = dataset_module.regression_rows(choice_df)
        specs = dataset_module.regression_specs(rows)
        module_by_dataset[piece_name] = dataset_module
        choice_dfs[piece_name] = choice_df
        regression_rows_by_dataset[piece_name] = rows
        regression_specs_by_dataset[piece_name] = specs
        human_ols_by_dataset[piece_name] = beta_core.fit_human_ols(rows, specs)
        print(
            f"{piece_name}: {choice_df['CASEID'].nunique():,} respondents | "
            f"{len(choice_df):,} tasks | {len(specs)} regression(s)"
        )
        display(beta_core.wide_ols_table(human_ols_by_dataset[piece_name]).round(4))

WU_DATASETS = piece_names_by_study["wu2025"]

# Table targets for `wu2025`: the 8 standardized OLS slopes fitted on Wu's choices, standing in for a published table
# (Wu's respondent choices y never enter the training loss or gradient)
target_std_betas_by_dataset: dict[str, dict[str, torch.Tensor]] = {}
for dataset_name in WU_DATASETS:
    dataset_ols = human_ols_by_dataset[dataset_name]
    dataset_targets = {}
    for group, spec in regression_specs_by_dataset[dataset_name].items():
        group_ols = dataset_ols[dataset_ols["model"] == group].set_index("term")
        target_values = group_ols.loc[spec.target_labels, "std_beta"].to_numpy(dtype="float64")
        dataset_targets[group] = torch.tensor(target_values, dtype=torch.float64)
        display(
            pd.Series(
                target_values,
                index=spec.target_labels,
                name=f"{dataset_name} | {group} fitted OLS target std_beta",
            ).round(4).to_frame()
        )
    target_std_betas_by_dataset[dataset_name] = dataset_targets

print(f"WU_DATASETS = {WU_DATASETS}")

# 2. Validators, Cached Step-0 Evaluation, & Cached `wu2025` Items

In [ ]:
VALIDATION_MAX_QUESTIONS = 1024
VALIDATION_MAX_CHARS = 1_000_000

validators = {
    dataset_name: PopulationValidator(
        beta_ols.DATASET_MODULES[dataset_name],
        choice_dfs[dataset_name],
        regression_specs_by_dataset[dataset_name],
        max_questions=VALIDATION_MAX_QUESTIONS,
        max_chars=VALIDATION_MAX_CHARS,
    )
    for dataset_name in VALIDATION_DATASETS
}
display(pd.DataFrame([v.describe() for v in validators.values()]).set_index("validator"))

In [ ]:
# Untrained Kev-9B evaluation (shared disk cache, keyed by MODEL_KEY)
step0_validation, step0_cache_report = validate_all_cached(
    kev_model,
    validators,
    INIT_TRAINABLE_STATE,
    model_key=MODEL_KEY,
    cache_dir=CACHE_DIR,
)
display(step0_cache_report)
display(validation_metrics_frame(step0_validation).round(5))

In [ ]:
item_modules = {dataset_name: module_by_dataset[dataset_name] for dataset_name in WU_DATASETS}
unified_items, items_cache_report = build_items_cached(
    kev_model, item_modules, choice_dfs, cache_dir=CACHE_DIR
)
display(items_cache_report)
display(items_summary(unified_items, regression_specs_by_dataset))

# 3. Shared Training Settings & Per-$\alpha$ Runner

Matches the Kev-0.8B sweep in `00-validate-kl-anchor-800m.ipynb` except `PEAK_LR = 5e-6` (the Kev-4B rate in `02-train-paper-4b.ipynb`) (`S = 40`, `3` epochs = `1,395` steps, `beta_lambda = 1.0`), with `kl_anchor_alpha = 0.005`. Validation runs every `50` steps on `kriner2026` (selection) and once per epoch (`465` steps) plus `best.pt` on held-out `grummon2025`.

In [ ]:
MAX_FITTING_MICRO_CHUNK = 16
RUN_MICRO_CHUNK_SEARCH = True

if RUN_MICRO_CHUNK_SEARCH:
    search_items = [item for ds in WU_DATASETS for item in unified_items[ds]]
    micro_chunk_search_df, MAX_FITTING_MICRO_CHUNK = find_max_micro_chunk(
        kev_model,
        search_items,
        INIT_TRAINABLE_STATE,
        max_chunk=41,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        candidates_per_size=3,
    )

    if MAX_FITTING_MICRO_CHUNK>1: MAX_FITTING_MICRO_CHUNK-=1 # for safety decrease microchunk by 1
    display(micro_chunk_search_df)

NOTEBOOK = "00-validate-kl-anchor-9b-alpha0.005"
BEST_METRIC = "kriner2026/beta_mse"
BEST_MODE = "min"

from_scratch = False

SAMPLE_SIZE = 40
PEAK_LR = 5e-6
NUM_EPOCHS = 3
BETA_LAMBDA = 1.0
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
SEED = 42
MICRO_CHUNK = MAX_FITTING_MICRO_CHUNK
P0_CHUNK = 32

EVAL_EVERY_BY_VALIDATOR = {
    "wu2025": 50,    # 28 evaluations across 1,395 steps (~8 min total)
    "kriner2026": 50,    # 28 evaluations across 1,395 steps (~8 min total)
    "grummon2025": 465,  # end of each Wu epoch (steps 465, 930, 1,395) + step 0 + best.pt
}

RUN_METADATA = {
    "notebook": NOTEBOOK,
    "model": MODEL_KEY,
    "item_datasets": ITEM_DATASETS,
    "validation_datasets": VALIDATION_DATASETS,
    "validator_kind": "fused",
}
runs = {}


def make_kl_anchor_config(kl_anchor_alpha: float) -> PaperTrainConfig:
    return PaperTrainConfig(
        ce_datasets=(),
        beta_datasets=WU_DATASETS,
        sample_size_by_dataset={ds: SAMPLE_SIZE for ds in WU_DATASETS},
        beta_lambda=BETA_LAMBDA,
        kl_anchor_alpha=kl_anchor_alpha,
        lr=PEAK_LR,
        num_epochs=NUM_EPOCHS,
        eval_every_by_validator=EVAL_EVERY_BY_VALIDATOR,
        lr_schedule="onecycle",
        weight_decay=WEIGHT_DECAY,
        micro_chunk=MICRO_CHUNK,
        p0_chunk=P0_CHUNK,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        grad_clip=GRAD_CLIP,
        seed=SEED,
    )


def train_kl_anchor_run(kl_anchor_alpha: float):
    slug = alpha_slug(kl_anchor_alpha)
    run_config = make_kl_anchor_config(kl_anchor_alpha)
    print(f"=== {slug} (beta_lambda={BETA_LAMBDA}, kl_anchor_alpha={kl_anchor_alpha}, lr={PEAK_LR}, epochs={NUM_EPOCHS}) ===")
    display(pd.Series(run_config.as_dict(), name=slug).to_frame())
    runs[slug] = run_paper_training(
        model=kev_model,
        items_by_dataset=unified_items,
        specs_by_dataset=regression_specs_by_dataset,
        target_std_betas_by_dataset=target_std_betas_by_dataset,
        validators=validators,
        step0_validation=step0_validation,
        init_snapshot=INIT_TRAINABLE_STATE,
        config=run_config,
        device=device,
        run_dir=RUNS_DIR / slug,
        best_metric=BEST_METRIC,
        best_mode=BEST_MODE,
        run_metadata={**RUN_METADATA, "kl_anchor_alpha": kl_anchor_alpha, "sweep_slug": slug},
        from_scratch=from_scratch,
    )
    return runs[slug]


## Run — `kl_anchor_alpha = 0.005`

In [ ]:
train_kl_anchor_run(0.005)